# Retinal Blood Vessel Segmentation (Morph-ViT) — Interactive Demo
### Fundamentals of Image Processing (FIP) Course Project

This notebook demonstrates the end-to-end processing pipeline:
1. **Spectral Green Channel Extraction & CLAHE Illumination Equalization**
2. **Multi-Directional Morphological Top-Hat Transform (12 orientations)**
3. **Morphology-Guided Vision Transformer (Morph-ViT) Inference**
4. **Post-Processing (Connected-Component Speckle Filtering)**
5. **Quantitative FOV-Masked Metric Computation**

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Add src to Python path
sys.path.insert(0, os.path.abspath('src'))

from dataset import load_image
from morphology import enhance_retinal_vessels, extract_green_channel, apply_clahe_manual, multi_directional_morphology
from models.morph_vit import MorphologyVisionTransformer
from postprocess import clean_isolated_specks
from metrics import compute_all_metrics

print('✓ Dependencies successfully imported!')

## Step 1: Load a Fundus Image and its Ground Truth

In [ ]:
IMG_SIZE = (576, 576)

# Select a sample from the DRIVE dataset
img_path = 'data/raw/training/images/21_training.tif'
fov_path = 'data/raw/training/mask/21_training_mask.gif'
lbl_path = 'data/raw/training/1st_manual/21_manual1.gif'

rgb = load_image(img_path, size=IMG_SIZE, mode='RGB')
fov = load_image(fov_path, size=IMG_SIZE, mode='L')
lbl = load_image(lbl_path, size=IMG_SIZE, mode='L')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb); axes[0].set_title('RGB Fundus (Input)'); axes[0].axis('off')
axes[1].imshow(fov, cmap='gray'); axes[1].set_title('FOV Mask'); axes[1].axis('off')
axes[2].imshow(lbl, cmap='gray'); axes[2].set_title('Ground Truth (Manual)'); axes[2].axis('off')
plt.tight_layout()
plt.show()

## Step 2: Classical Image Processing Pipeline
Here we execute the domain-specific physics-based filters:
1. **Green channel extraction** (hemoglobin peak absorption)
2. **CLAHE** (local illumination leveling)
3. **Multi-directional Top-Hat transform** (12 orientations $0^\circ$ to $165^\circ$)

In [ ]:
green = extract_green_channel(rgb)
clahe = apply_clahe_manual(green, clip_limit=2.0)
morph_vessels = multi_directional_morphology(clahe, length=15)
morph_norm = (morph_vessels.astype(np.float32) / 255.0) * (fov > 0).astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(green, cmap='gray'); axes[0].set_title('1. Green Channel'); axes[0].axis('off')
axes[1].imshow(clahe, cmap='gray'); axes[1].set_title('2. After CLAHE'); axes[1].axis('off')
axes[2].imshow(morph_vessels, cmap='hot'); axes[2].set_title('3. Multi-Dir Top-Hat'); axes[2].axis('off')
axes[3].imshow(morph_norm, cmap='hot'); axes[3].set_title('4. Normalized inside FOV'); axes[3].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Deep Vision Transformer (Morph-ViT) Inference

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = 'outputs/checkpoints/best_model.pth'

# Load checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model = MorphologyVisionTransformer(
    in_channels=4, patch_size=16, embed_dim=128, depth=4, num_heads=4, mlp_ratio=2.0, dropout=0.0
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Construct 4-channel input: [R, G, B, Morph]
rgb_t = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1)
morph_t = torch.from_numpy(morph_norm).unsqueeze(0)
x = torch.cat([rgb_t, morph_t], dim=0).unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    logits = model(x)
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    prob = prob * (fov > 0).astype(np.float32)

print(f'✓ Forward pass complete! Output probability range: [{prob.min():.4f}, {prob.max():.4f}]')

## Step 4: Calibrated Thresholding ($T=0.65$) + Morphological Speck Clean-up

In [ ]:
# 1. Uncalibrated threshold
bin_raw = (prob >= 0.50).astype(np.uint8)

# 2. Calibrated threshold
bin_calib = (prob >= 0.65).astype(np.uint8)

# 3. Calibrated threshold + Morphological connected-component speck removal
bin_clean = clean_isolated_specks(bin_calib, min_size=10, fov_mask=fov)

# Create green vessel overlay on original RGB
overlay = rgb.copy().astype(np.float32)
heat = np.zeros_like(rgb, dtype=np.float32)
heat[..., 1] = bin_clean * 255
overlay = (0.5 * overlay + 0.5 * heat).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(prob, cmap='hot'); axes[0].set_title('Predicted Probability'); axes[0].axis('off')
axes[1].imshow(bin_raw, cmap='gray'); axes[1].set_title('Raw Threshold (T=0.50)'); axes[1].axis('off')
axes[2].imshow(bin_clean, cmap='gray'); axes[2].set_title('Optimized (T=0.65 + Speck Filter)'); axes[2].axis('off')
axes[3].imshow(overlay); axes[3].set_title('Final Vessel Overlay'); axes[3].axis('off')
plt.tight_layout()
plt.show()

## Step 5: Quantitative Evaluation Inside Field of View (FOV)

In [ ]:
m_raw = compute_all_metrics(y_true=lbl, y_pred_probs=bin_raw.astype(np.float32), fov_mask=fov, threshold=0.5)
m_opt = compute_all_metrics(y_true=lbl, y_pred_probs=bin_clean.astype(np.float32), fov_mask=fov, threshold=0.5)

print('='*60)
print(f"{'Metric':<20} {'Raw (T=0.50)':<18} {'Optimized (T=0.65)':<18}")
print('='*60)
for k in ['Accuracy', 'Dice', 'IoU', 'Precision', 'Sensitivity', 'Specificity']:
    print(f"{k:<20} {m_raw[k]*100:<17.2f}% {m_opt[k]*100:<17.2f}%")
print('='*60)